In [2]:
!pip install openmeteo_requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.5/772.5 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 71.5 MB/s eta 0:00:00
  Attempting uninstall: flatbuffers
    Found existing installation: flatbuffers 25.12.19
    Uninstalling flatbuffers-25.12.19:
      Successfully uninstalled flatbuffers-25.12.19


In [3]:
from datetime import datetime
import openmeteo_requests

In [16]:
# iterator for smooth speed acceleration, ensures the maximum threshold is never violated
class IncreaseSpeed:
    def __init__(self, start_speed: int, limit_speed: int, rate=10):
        self.current = start_speed
        self.upper_bound = limit_speed
        self.step = abs(rate) if rate != 0 else 10

    def __iter__(self):
        return self

    def __next__(self):
        if self.current >= self.upper_bound:
            raise StopIteration

        self.current = min(self.current + self.step, self.upper_bound)
        return self.current

In [17]:
# iterator for systematic braking, prevents speed values from dropping below the lower boundary
class DecreaseSpeed:
    def __init__(self, start_speed: int, limit_speed: int, rate=10):
        self.current = start_speed
        self.lower_bound = limit_speed
        self.step = abs(rate) if rate != 0 else 10

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= self.lower_bound:
            raise StopIteration

        self.current = max(self.current - self.step, self.lower_bound)
        return self.current

In [18]:
# represents a customizable vehicle state tracker, tracks active cars on the road and integrates with weather services.
class Car:
    cars_on_road = 0  # Class variable tracking vehicles on the road

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max(max_speed, 0)
        self.current_speed = min(max(current_speed, 0), self.max_speed)
        self.state = 'on the road' if self.current_speed > 0 else 'parking'

        if self.state == 'on the road':
            Car.cars_on_road += 1

    def _update_road_state(self):
        if self.state == 'parking' and self.current_speed > 0:
            self.state = 'on the road'
            Car.cars_on_road += 1

    def accelerate(self, upper_border=None, step=10):
        initial = self.current_speed
        is_gradual = upper_border is not None and self.current_speed <= upper_border <= self.max_speed
        target = upper_border if is_gradual else self.max_speed

        engine_controller = IncreaseSpeed(self.current_speed, target, step)

        if is_gradual:
            for speed_milestone in engine_controller:
                print(f'INFO: Speed increases by {speed_milestone - self.current_speed}')
                self.current_speed = speed_milestone
                self._update_road_state()
        else:
            try:
                speed_milestone = next(engine_controller)
                print(f'INFO: Speed increases by {speed_milestone - self.current_speed}')
                self.current_speed = speed_milestone
                self._update_road_state()
            except StopIteration:
                pass

        log_message = f'INFO: The speed of this car has been increased from {initial} to {self.current_speed}'
        print(log_message)
        return log_message

In [19]:
# represents a customizable vehicle state tracker, tracks active cars on the road and integrates with weather services.
class Car:
    cars_on_road = 0

    def __init__(self, max_speed: int, current_speed=0):
        self.max_speed = max(max_speed, 0)
        self.current_speed = min(max(current_speed, 0), self.max_speed)
        self.state = 'on the road' if self.current_speed > 0 else 'parking'

        if self.state == 'on the road':
            Car.cars_on_road += 1

    def _update_road_state(self):
        if self.state == 'parking' and self.current_speed > 0:
            self.state = 'on the road'
            Car.cars_on_road += 1

    def accelerate(self, upper_border=None, step=10):
        initial = self.current_speed
        is_gradual = upper_border is not None and self.current_speed <= upper_border <= self.max_speed
        target = upper_border if is_gradual else self.max_speed

        engine_controller = IncreaseSpeed(self.current_speed, target, step)

        if is_gradual:
            for speed_milestone in engine_controller:
                print(f'INFO: Speed increases by {speed_milestone - self.current_speed}')
                self.current_speed = speed_milestone
                self._update_road_state()
        else:
            try:
                speed_milestone = next(engine_controller)
                print(f'INFO: Speed increases by {speed_milestone - self.current_speed}')
                self.current_speed = speed_milestone
                self._update_road_state()
            except StopIteration:
                pass

        log_message = f'INFO: The speed of this car has been increased from {initial} to {self.current_speed}'
        print(log_message)
        return log_message

    def brake(self, lower_border=None, step=10):
        initial = self.current_speed
        is_gradual = lower_border is not None and 0 <= lower_border <= self.current_speed
        target = lower_border if is_gradual else 0

        brake_controller = DecreaseSpeed(self.current_speed, target, step)

        if is_gradual:
            for speed_milestone in brake_controller:
                print(f'INFO: Speed decreases by {self.current_speed - speed_milestone}')
                self.current_speed = speed_milestone
        else:
            try:
                speed_milestone = next(brake_controller)
                print(f'INFO: Speed decreases by {self.current_speed - speed_milestone}')
                self.current_speed = speed_milestone
            except StopIteration:
                pass

        log_message = f'INFO: The speed of this car has been decreased from {initial} to {self.current_speed}'
        print(log_message)
        return log_message

# acts on instance attributes (`self.state`, `self.current_speed`)
    def parking(self):
        if self.state == 'parking':
            log_message = 'INFO: This car is already in the parking'
            print(log_message)
            return log_message

        self.brake(0)
        self.state = 'parking'
        Car.cars_on_road -= 1
        log_message = 'Parking the car...'
        print(log_message)
        return log_message

    @classmethod
    # accesses and monitors the state of class-wide variables (`cls.cars_on_road`)
    def total_cars(cls):

        return cls.cars_on_road

    @staticmethod
    def show_weather():
        meteo_client = openmeteo_requests.Client()
        forecast_url = 'https://api.open-meteo.com/v1/forecast'
        request_params = {
            'latitude': 59.9386,
            'longitude': 30.3141,
            'current': ['temperature_2m', 'apparent_temperature', 'rain', 'wind_speed_10m'],
            'wind_speed_unit': 'ms',
            'timezone': 'Europe/Moscow'
        }

        api_payload = meteo_client.weather_api(forecast_url, params=request_params)[0]
        live_metrics = api_payload.Current()

        t_2m = live_metrics.Variables(0).Value()
        t_app = live_metrics.Variables(1).Value()
        precipitation = live_metrics.Variables(2).Value()
        w_speed = live_metrics.Variables(3).Value()

        timestamp = datetime.fromtimestamp(live_metrics.Time() + api_payload.UtcOffsetSeconds())
        tz_tag = api_payload.TimezoneAbbreviation().decode()

        print(f"Current time: {timestamp} {tz_tag}")
        print(f'Current temperature: {round(t_2m, 0)} C')
        print(f'Current apparent_temperature: {round(t_app, 0)} C')
        print(f'Current rain: {precipitation} mm')
        print(f'Current wind_speed: {round(w_speed, 1)} m/s')

In [20]:
car1 = Car(100, 20)
car2 = Car(60, 30)
car3 = Car(100, 0)
print(f"Total cars on road: {Car.total_cars()}")

Total cars on road: 2


In [21]:
car1.accelerate(100)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 20 to 100


'INFO: The speed of this car has been increased from 20 to 100'

In [22]:
car2.accelerate(50)
print("Speed of car 1:", car1.current_speed)
print("Speed of car 2:", car2.current_speed)

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 30 to 50
Speed of car 1: 100
Speed of car 2: 50


In [23]:
car1.brake(10)

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 100 to 10


'INFO: The speed of this car has been decreased from 100 to 10'

In [24]:
car2.brake(0)
print("Total cars on road:", Car.total_cars())
car2.parking()
print("Total cars on road:", Car.total_cars())

INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: Speed decreases by 10
INFO: The speed of this car has been decreased from 50 to 0
Total cars on road: 2
INFO: The speed of this car has been decreased from 0 to 0
Parking the car...
Total cars on road: 1


In [25]:
car3.accelerate(80)
car3.show_weather()
print("Total cars on road:", Car.total_cars())

INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 80
Current time: 2026-06-11 09:15:00 GMT+3
Current temperature: 16.0 C
Current apparent_temperature: 15.0 C
Current rain: 0.0 mm
Current wind_speed: 1.4 m/s
Total cars on road: 2


In [26]:
car2.accelerate(10)
print("Total cars on road:", Car.total_cars())
Car.show_weather()

INFO: Speed increases by 10
INFO: The speed of this car has been increased from 0 to 10
Total cars on road: 3
Current time: 2026-06-11 09:15:00 GMT+3
Current temperature: 16.0 C
Current apparent_temperature: 15.0 C
Current rain: 0.0 mm
Current wind_speed: 1.4 m/s
